# 02 · Un proyecto de ML reproducible

**Módulo 1 · Sesión 2** — Ciclo de vida de un proyecto de Machine Learning

## Objetivos

Un modelo que no se puede volver a producir no sirve, por bueno que sea. Este notebook
convierte esa afirmación en práctica concreta:

1. Recorrer el ciclo de vida de un proyecto de ML y ubicar cada sesión del curso en él.
2. Organizar un proyecto para que otra persona pueda ejecutarlo.
3. Los cuatro niveles de reproducibilidad: semilla, entorno, datos y modelo.
4. Qué versionar con Git y por qué los datos necesitan otra herramienta (DVC).

> **Nota.** Este notebook crea archivos en una carpeta temporal del sistema y la borra al
> final. No ensucia el repositorio.

## Paquetes

`pandas`, `numpy`, `scikit-learn`, `joblib`. Todo de la biblioteca estándar lo demás.

In [ ]:
import hashlib
import json
import platform
import shutil
import sys
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

SEMILLA = 42

# Carpeta de trabajo temporal para los ejemplos de este notebook.
TRABAJO = Path(tempfile.mkdtemp(prefix="ml_sesion02_"))
print(f"Carpeta temporal: {TRABAJO}")

## 1. El ciclo de vida

Un proyecto de Machine Learning no es una línea recta: es un ciclo al que se vuelve muchas
veces. Estas son sus etapas, y dónde las trabaja este curso.

| # | Etapa | Qué se decide | Sesiones |
|---|---|---|---|
| 1 | **Definición del problema** | Qué se predice, qué métrica define el éxito, qué restricciones hay | S1, S2 |
| 2 | **Recolección y preparación** | De dónde salen los datos, cómo se limpian | S4 |
| 3 | **Análisis exploratorio (EDA)** | Qué hay realmente en los datos, qué se puede esperar | S4 |
| 4 | **Construcción y entrenamiento** | Qué características, qué algoritmo, qué hiperparámetros | S5–S13 |
| 5 | **Evaluación** | Si el modelo sirve, comparado contra qué | S8, S9 |
| 6 | **Despliegue** | Cómo lo consume el resto del mundo | S14 |
| 7 | **Monitoreo y mantenimiento** | Si sigue sirviendo dentro de seis meses | S14 |

### La etapa 1 es la que decide el proyecto

El error más caro de un proyecto de ML no es elegir mal el algoritmo: es resolver
bien el problema equivocado. Antes de tocar los datos hay que responder:

- **¿Cuál es la decisión que este modelo va a apoyar?** Si no cambia ninguna decisión, el
  modelo no tiene por qué existir.
- **¿Cuál es la métrica de negocio?** No "accuracy": cuántos estudiantes en riesgo
  detectamos a tiempo, cuánto cuesta una falsa alarma.
- **¿Cuál es la referencia actual?** Si hoy alguien decide con una regla simple, esa es la
  marca a superar.
- **¿Qué información estará disponible en el momento de predecir?** Esta pregunta previene
  la fuga de datos (S5): si una variable solo se conoce *después* del hecho que quieres
  predecir, no puedes usarla.

> **Métrica técnica ≠ métrica de negocio.** Un modelo con MAE de 0.26 puntos de nota no
> dice nada por sí solo. La pregunta es: ¿con ese error, la universidad puede identificar a
> tiempo a quién ofrecer tutorías? Esa traducción es responsabilidad de quien modela.

## 2. Estructura de un proyecto

La estructura concreta importa menos que el hecho de tener una y respetarla. Esta es
suficiente para casi todo, y es la que se espera en el proyecto integrador:

```
proyecto/
├── README.md              # qué hace, cómo se ejecuta
├── requirements.txt       # dependencias con versión
├── .gitignore
├── datos/
│   ├── crudos/            # datos originales: NUNCA se modifican
│   └── procesados/        # resultado del preprocesamiento (se regenera)
├── notebooks/             # exploración; el código estable emigra a src/
├── src/
│   ├── preparar_datos.py
│   ├── entrenar.py
│   └── evaluar.py
├── modelos/               # artefactos entrenados
└── resultados/            # métricas, figuras
```

Dos reglas que ahorran muchos disgustos:

1. **Los datos crudos son de solo lectura.** Toda transformación produce un archivo nuevo.
   Si sobrescribes el original, perdiste la capacidad de rehacer el trabajo.
2. **Todo lo derivado debe poder regenerarse con un comando.** Si no, terminarás con
   `datos_final_v3_bueno_ESTE.csv`.

In [ ]:
for carpeta in ["datos/crudos", "datos/procesados", "src", "modelos", "resultados"]:
    (TRABAJO / carpeta).mkdir(parents=True, exist_ok=True)

print("Estructura creada:")
for ruta in sorted(TRABAJO.rglob("*")):
    print("  " + str(ruta.relative_to(TRABAJO)).replace("\\", "/") + "/")

## 3. Reproducibilidad nivel 1 — la semilla

Muchísimos pasos de un flujo de ML son aleatorios: la partición train/test, la
inicialización de pesos, el remuestreo de un Random Forest. Sin semilla fija, dos
ejecuciones dan resultados distintos y no puedes saber si una mejora vino de tu cambio o
del azar.

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")
caracteristicas = ["promedio_anterior", "horas_estudio_semana", "asistencia_pct", "trabaja"]
X, y = datos[caracteristicas], datos["nota_final"]

print("SIN semilla fija — tres ejecuciones del mismo código:")
for i in range(3):
    X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2)
    modelo = RandomForestRegressor(n_estimators=50)
    modelo.fit(X_ent, y_ent)
    mae = mean_absolute_error(y_pru, modelo.predict(X_pru))
    print(f"  ejecución {i + 1}: MAE = {mae:.4f}")

Tres números distintos para el mismo código. Si tu compañero reporta 0.28 y tú 0.31, no
tienen forma de saber si el modelo cambió.

In [ ]:
print("CON semilla fija — tres ejecuciones del mismo código:")
for i in range(3):
    X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)
    modelo = RandomForestRegressor(n_estimators=50, random_state=SEMILLA)
    modelo.fit(X_ent, y_ent)
    mae = mean_absolute_error(y_pru, modelo.predict(X_pru))
    print(f"  ejecución {i + 1}: MAE = {mae:.4f}")

Idénticos. **Regla del curso: toda función con `random_state` lo lleva puesto.**

> **Ojo con el matiz.** La semilla no hace que un modelo sea bueno, hace que sea
> *comparable*. Un resultado que depende fuertemente de la semilla es un resultado frágil, y
> eso es información valiosa: en la sesión 8 mediremos esa variabilidad a propósito con
> validación cruzada, en vez de esconderla detrás de una sola partición afortunada.

## 4. Reproducibilidad nivel 2 — el entorno

El mismo código con otra versión de scikit-learn puede dar otro resultado. Registrar el
entorno junto a los resultados es parte del experimento.

In [ ]:
entorno = {
    "python": sys.version.split()[0],
    "sistema": platform.system(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
}

ruta_entorno = TRABAJO / "resultados" / "entorno.json"
ruta_entorno.write_text(json.dumps(entorno, indent=2), encoding="utf-8")

print(json.dumps(entorno, indent=2))

En el proyecto integrador esto se resuelve con `requirements.txt` (o `environment.yml`) con
versiones fijadas. En la sesión 14 veremos cómo **MLflow** guarda esto automáticamente en
cada experimento, junto con los parámetros y las métricas.

## 5. Reproducibilidad nivel 3 — los datos

"Usé el dataset de estudiantes" no es una descripción suficiente: ¿cuál versión? Un
**hash** resuelve el problema: es una huella digital del archivo. Si cambia un solo byte,
el hash cambia por completo.

In [ ]:
def hash_archivo(ruta, algoritmo="sha256"):
    h = hashlib.new(algoritmo)
    with open(ruta, "rb") as fh:
        for bloque in iter(lambda: fh.read(65536), b""):
            h.update(bloque)
    return h.hexdigest()


ruta_datos = Path("../datos/rendimiento-estudiantes.csv")
huella = hash_archivo(ruta_datos)

print(f"Archivo: {ruta_datos.name}")
print(f"Tamaño:  {ruta_datos.stat().st_size:,} bytes")
print(f"SHA-256: {huella}")

Comprobemos que detecta cualquier modificación, por pequeña que sea.

In [ ]:
copia = TRABAJO / "datos" / "crudos" / "copia.csv"
shutil.copy(ruta_datos, copia)
print(f"Copia idéntica:   {hash_archivo(copia) == huella}")

# Modificamos un único valor.
alterado = pd.read_csv(copia)
alterado.loc[0, "nota_final"] = alterado.loc[0, "nota_final"] + 0.01
alterado.to_csv(copia, index=False, encoding="utf-8")

print(f"Tras cambiar una nota: {hash_archivo(copia) == huella}")
print(f"Nuevo hash:            {hash_archivo(copia)[:32]}...")

## 6. Reproducibilidad nivel 4 — el modelo entrenado

Entrenar puede tardar horas. El modelo ajustado es un **artefacto** que se guarda y se
vuelve a cargar, y es exactamente lo que se despliega en la sesión 14.

In [ ]:
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)
modelo = RandomForestRegressor(n_estimators=100, random_state=SEMILLA)
modelo.fit(X_ent, y_ent)

ruta_modelo = TRABAJO / "modelos" / "modelo-v1.joblib"
joblib.dump(modelo, ruta_modelo)

print(f"Modelo guardado: {ruta_modelo.name} ({ruta_modelo.stat().st_size / 1024:.0f} KB)")

In [ ]:
modelo_cargado = joblib.load(ruta_modelo)

pred_original = modelo.predict(X_pru)
pred_cargado = modelo_cargado.predict(X_pru)

print(f"¿Predicciones idénticas? {np.array_equal(pred_original, pred_cargado)}")
print(f"MAE del modelo cargado:  {mean_absolute_error(y_pru, pred_cargado):.4f}")

### La ficha del experimento

Con las cuatro piezas juntas —semilla, entorno, hash de datos y artefacto— cualquiera puede
reconstruir exactamente este resultado.

In [ ]:
ficha = {
    "experimento": "linea-base-random-forest",
    "semilla": SEMILLA,
    "datos": {"archivo": ruta_datos.name, "sha256": huella},
    "modelo": {
        "tipo": type(modelo).__name__,
        "n_estimators": modelo.n_estimators,
        "artefacto": ruta_modelo.name,
    },
    "caracteristicas": caracteristicas,
    "metricas": {"mae_prueba": round(float(mean_absolute_error(y_pru, pred_cargado)), 4)},
    "entorno": entorno,
}

ruta_ficha = TRABAJO / "resultados" / "experimento.json"
ruta_ficha.write_text(json.dumps(ficha, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(ficha, indent=2, ensure_ascii=False))

Escribir esto a mano se vuelve insostenible en cuanto pruebas veinte configuraciones. Por
eso existe MLflow, que lo hace por ti (sesión 14). Pero conviene haberlo hecho a mano una
vez para entender qué está guardando la herramienta y por qué.

## 7. Git: qué se versiona

Git es excelente para archivos de texto pequeños y pésimo para archivos binarios grandes.
En un proyecto de ML eso se traduce en:

| Se versiona en Git | No se versiona |
|---|---|
| Código (`.py`, `.ipynb`) | Datasets grandes |
| Configuración, `requirements.txt` | Modelos entrenados (`.joblib`, `.pkl`) |
| Documentación, `README.md` | Salidas regenerables |
| Datasets pequeños (< 1 MB) | Credenciales y secretos |

Los comandos mínimos para el proyecto integrador:

```bash
git init
git add .
git commit -m "Linea base: random forest con MAE 0.26"
git log --oneline
```

### Notebooks y Git

Un `.ipynb` es un JSON que guarda también las salidas. Si haces commit con las salidas
puestas, cada ejecución genera un diff enorme e ilegible. **Limpia las salidas antes de
hacer commit** (`Kernel → Restart & Clear Output`). Es una de las convenciones de este
repositorio.

## 8. DVC: control de versiones para datos

¿Y el dataset de 2 GB? Git no puede con él. **DVC** (Data Version Control) resuelve esto:
guarda el archivo grande en un almacenamiento aparte y deja en Git solo un archivo de texto
diminuto —un puntero con el hash— que sí se versiona.

La idea es exactamente la del hash de la sección 5, automatizada.

```bash
dvc init                          # dentro de un repo git ya existente
dvc add datos/crudos/dataset.csv  # crea dataset.csv.dvc y lo ignora en git
git add datos/crudos/dataset.csv.dvc .gitignore
git commit -m "Agrega dataset v1"

dvc remote add -d almacen /ruta/o/s3://bucket
dvc push                          # sube los datos al almacenamiento
```

Y quien clone el repositorio recupera los datos correspondientes a ese commit:

```bash
git clone <repo>
dvc pull
```

El resultado es que `git checkout` de un commit antiguo, seguido de `dvc pull`, te devuelve
**el código y los datos exactos** de aquel momento. Eso es lo que hace auditable un proyecto
de ML.

## 9. Limpieza

In [ ]:
shutil.rmtree(TRABAJO)
print(f"Carpeta temporal eliminada: {not TRABAJO.exists()}")

## Lista de verificación

Antes de dar por terminado cualquier experimento del curso:

- [ ] ¿Toda fuente de aleatoriedad tiene semilla fija?
- [ ] ¿El notebook corre de principio a fin en un kernel reiniciado?
- [ ] ¿Las rutas son relativas, no absolutas?
- [ ] ¿Están registradas las versiones de los paquetes?
- [ ] ¿Se sabe exactamente qué versión de los datos se usó?
- [ ] ¿El modelo entrenado está guardado como artefacto?
- [ ] ¿Hay una referencia trivial contra la cual comparar?
- [ ] ¿El código está en Git y los datos en DVC?

## Para practicar

1. Ejecuta la sección 3 cambiando `SEMILLA` a 7. ¿Cuánto cambia el MAE? ¿Qué te dice sobre
   la fiabilidad de reportar un solo número?
2. Escribe una función `verificar_datos(ruta, hash_esperado)` que lance un error si el
   dataset no es el esperado. Es una defensa barata contra un fallo silencioso.
3. Compara el tamaño del `.joblib` de un `RandomForestRegressor` con 100 y con 500 árboles.
   ¿Qué implica para el despliegue?
4. Toma el notebook 01 y reorganízalo en la estructura de proyecto de la sección 2, con
   `src/entrenar.py` ejecutable desde la terminal.